In [1]:
import time
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score,
                             confusion_matrix, mean_squared_error,
                             r2_score, mean_absolute_error)

train = pd.read_pickle("train.pkl")
test = pd.read_pickle("test.pkl")

feature_cols = [c for c in train.columns if c != "isFraud"]
X_train_raw, y_train = train[feature_cols], train["isFraud"].values
X_test_raw, y_test = test[feature_cols], test["isFraud"].values
print(X_train_raw.shape, X_test_raw.shape)

(5090096, 13) (1272524, 13)


In [2]:
money_cols = ["amount", "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest",
              "newbalanceDest", "errorBalanceOrig", "errorBalanceDest"]

def signed_log(df):
    df = df.copy()
    for c in money_cols:
        df[c] = np.sign(df[c]) * np.log1p(np.abs(df[c]))
    return df

scaler = StandardScaler()
X_train = scaler.fit_transform(signed_log(X_train_raw)).astype("float32")
X_test = scaler.transform(signed_log(X_test_raw)).astype("float32")
print(X_train.shape, X_train.dtype)

(5090096, 13) float32


In [3]:
def evaluate(name, y_true, prob, train_time, threshold=0.5):
    pred = (prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred)
    print(name, "confusion matrix:\n", cm)
    return {
        "model": name,
        "RMSE": float(np.sqrt(mean_squared_error(y_true, prob))),
        "R2": float(r2_score(y_true, prob)),
        "MAE": float(mean_absolute_error(y_true, prob)),
        "train_time_s": train_time,
        "precision": precision_score(y_true, pred),
        "recall": recall_score(y_true, pred),
        "F1": f1_score(y_true, pred),
        "ROC_AUC": roc_auc_score(y_true, prob),
        "PR_AUC": average_precision_score(y_true, prob),
    }

In [4]:
import tensorflow as tf
from tensorflow import keras

def build_model(n_features):
    model = keras.Sequential([
        keras.layers.Input(shape=(n_features,)),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="binary_crossentropy",
                  metrics=[keras.metrics.AUC(curve="PR", name="pr_auc")])
    return model

# quick test: 300k rows, 2 epochs
idx = np.random.RandomState(42).choice(len(X_train), 300_000, replace=False)
m = build_model(X_train.shape[1])
m.fit(X_train[idx], y_train[idx], epochs=2, batch_size=2048,
      class_weight={0: 1.0, 1: 28.0}, verbose=1)

Epoch 1/2
147/147 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.1411 - pr_auc: 0.1352
Epoch 2/2
147/147 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0146 - pr_auc: 0.9763


In [5]:
n_neg, n_pos = (y_train == 0).sum(), (y_train == 1).sum()
weight_pos = float(np.sqrt(n_neg / n_pos))   # about 28
print("class weight for fraud:", weight_pos)

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_pr_auc", mode="max", patience=2, restore_best_weights=True)

keras_model = build_model(X_train.shape[1])
start = time.time()
keras_model.fit(X_train, y_train, epochs=10, batch_size=4096,
                validation_split=0.1,
                class_weight={0: 1.0, 1: weight_pos},
                callbacks=[early_stop], verbose=1)
keras_time = time.time() - start
print("Training time (s):", keras_time)

class weight for fraud: 27.816330628238557
Epoch 1/10
1119/1119 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - loss: 0.0333 - pr_auc: 0.8436 - val_loss: 7.1176e-04 - val_pr_auc: 0.9902
Epoch 2/10
1119/1119 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - loss: 0.0021 - pr_auc: 0.9950 - val_loss: 4.2257e-04 - val_pr_auc: 0.9917
Epoch 3/10
1119/1119 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - loss: 0.0015 - pr_auc: 0.9961 - val_loss: 3.0891e-04 - val_pr_auc: 0.9922
Epoch 4/10
1119/1119 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - loss: 0.0012 - pr_auc: 0.9967 - val_loss: 2.4426e-04 - val_pr_auc: 0.9932
Epoch 5/10
1119/1119 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - loss: 0.0011 - pr_auc: 0.9970 - val_loss: 4.0909e-04 - val_pr_auc: 0.9946
Epoch 6/10
1119/1119 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - loss: 0.0010 - pr_auc: 0.9970 - val_loss: 2.0040e-04 - val_pr_auc: 0.9937
Epoch 7/10
1119/1119 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - loss: 9.5758e-04 - pr_auc: 0.9972 - val_loss: 4.9223e-04 - val_pr_auc: 0.9943
Training time (s): 60.933024644851685


In [6]:
keras_prob = keras_model.predict(X_test, batch_size=8192).ravel()
keras_row = evaluate("Keras MLP", y_test, keras_prob, keras_time)
print(keras_row)

pd.DataFrame([keras_row]).to_csv("keras_results.csv", index=False)

156/156 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
Keras MLP confusion matrix:
 [[1270865      16]
 [      4    1639]]
{'model': 'Keras MLP', 'RMSE': 0.00436510496733106, 'R2': 0.9852232336997986, 'MAE': 0.000316909805405885, 'train_time_s': 60.933024644851685, 'precision': 0.9903323262839879, 'recall': 0.9975654290931223, 'F1': 0.9939357186173439, 'ROC_AUC': 0.9998943041550356, 'PR_AUC': 0.9986151732378645}


In [7]:
from fastai.tabular.all import *
import torch

rng = np.random.RandomState(42)
idx = rng.choice(len(X_train), 1_000_000, replace=False)
idx.sort()

df_tr = pd.DataFrame(X_train[idx], columns=feature_cols)
df_tr["isFraud"] = y_train[idx]
df_te = pd.DataFrame(X_test, columns=feature_cols)

n_valid = 100_000
valid_idx = list(range(len(df_tr) - n_valid, len(df_tr)))

dls = TabularDataLoaders.from_df(
    df_tr, path=".", y_names="isFraud", cont_names=feature_cols,
    procs=[], y_block=CategoryBlock(), valid_idx=valid_idx,
    bs=4096, num_workers=0)

C:\Users\Windows\venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\Windows\venv\Lib\site-packages\fastai\torch_core.py:155: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:214.)
  else as_tensor(x.values, **kwargs) if isinstance(x, (pd.Series, pd.DataFrame))


In [8]:
w = torch.tensor([1.0, weight_pos])
learn = tabular_learner(dls, layers=[64, 32],
                        loss_func=CrossEntropyLossFlat(weight=w),
                        metrics=accuracy)

start = time.time()
learn.fit_one_cycle(3, 1e-3)
fastai_time = time.time() - start
print("Training time (s):", fastai_time)

epoch,train_loss,valid_loss,accuracy,time
0,0.300320,0.123535,0.999630,00:03
1,0.025140,0.015821,0.999980,00:03
2,0.010268,0.011843,0.999980,00:03


Training time (s): 11.119343280792236


In [9]:
w = torch.tensor([1.0, weight_pos])
learn = tabular_learner(dls, layers=[64, 32],
                        loss_func=CrossEntropyLossFlat(weight=w),
                        metrics=accuracy)

start = time.time()
learn.fit_one_cycle(3, 1e-3)
fastai_time = time.time() - start
print("Training time (s):", fastai_time)

epoch,train_loss,valid_loss,accuracy,time
0,0.274558,0.117932,0.999580,00:03
1,0.024403,0.015724,0.999960,00:03
2,0.010088,0.011795,0.999980,00:03


Training time (s): 11.141610145568848


In [10]:
test_dl = learn.dls.test_dl(df_te)
probs, _ = learn.get_preds(dl=test_dl)
fastai_prob = probs[:, 1].numpy()

fastai_row = evaluate("fastai Tabular", y_test, fastai_prob, fastai_time)
print(fastai_row)

pd.DataFrame([fastai_row]).to_csv("fastai_results.csv", index=False)

fastai Tabular confusion matrix:
 [[1270832      49]
 [      9    1634]]
{'model': 'fastai Tabular', 'RMSE': 0.011738088788892077, 'R2': 0.8931475877761841, 'MAE': 0.009665150195360184, 'train_time_s': 11.141610145568848, 'precision': 0.9708853238265003, 'recall': 0.9945222154595252, 'F1': 0.9825616355983163, 'ROC_AUC': 0.9990060762134678, 'PR_AUC': 0.9976759412691294}


In [11]:
print(pd.read_csv("keras_results.csv"))
print(pd.read_csv("fastai_results.csv"))

       model      RMSE        R2       MAE  train_time_s  precision    recall  \
0  Keras MLP  0.004365  0.985223  0.000317     60.933025   0.990332  0.997565   

         F1   ROC_AUC    PR_AUC  
0  0.993936  0.999894  0.998615  
            model      RMSE        R2       MAE  train_time_s  precision  \
0  fastai Tabular  0.011738  0.893148  0.009665      11.14161   0.970885   

     recall        F1   ROC_AUC    PR_AUC  
0  0.994522  0.982562  0.999006  0.997676  
